In [ ]:
%pip install xlrd

In [ ]:
from google.colab import files

uploaded = files.upload()
filename = next(iter(uploaded))
print("Uploaded file:", filename)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

# The dataset has an introductory row, so header=1 uses the second row as column names.
df = pd.read_excel(filename, header=1)
df.columns = df.columns.astype(str).str.strip()

target_col = "default payment next month"

print("Dataset loaded successfully!")
print("Dataset shape:", df.shape)
display(df.head())

In [ ]:
# Basic data-quality checks
print("Dataset shape:", df.shape)

print("\nTotal missing values:", df.isnull().sum().sum())
print("Duplicate rows:", df.duplicated().sum())

print("\nData types:")
print(df.dtypes)

print("\nColumn names:")
print(df.columns.tolist())

# Confirm that the target column exists and contains the expected two classes.
print("\nTarget values:")
print(df[target_col].value_counts().sort_index())

## Question 1: What proportion of customers defaulted compared with those who did not?

In [ ]:
# Count customers in each target category
default_counts = df[target_col].value_counts().reindex([0, 1])

# Calculate the percentage in each category
default_percentages = df[target_col].value_counts(normalize=True).reindex([0, 1]) * 100

# Build a clearly labelled summary table
results = pd.DataFrame({
    "Customer Group": ["No Default", "Default"],
    "Number of Customers": default_counts.values,
    "Percentage (%)": default_percentages.values
})

display(results)

In [ ]:
plt.figure(figsize=(8, 5))

sns.barplot(
    x=["No Default", "Default"],
    y=default_percentages.values
)

plt.title("Distribution of Customers by Default Status")
plt.xlabel("Default Status (0 = No Default, 1 = Default)")
plt.ylabel("Customers (%)")
plt.ylim(0, 100)

for position, value in enumerate(default_percentages.values):
    plt.text(position, value + 1, f"{value:.2f}%", ha="center")

plt.tight_layout()
plt.show()

### Interpretation of Question 1

The analysis of 30,000 customer records shows that 23,364 customers (77.88%) did not default, while 6,636 customers (22.12%) defaulted on their credit card payments in the following month. Although most customers did not default, the proportion who did default is substantial. This makes it important for financial institutions to identify factors associated with default and strengthen credit-risk assessment.

## Question 2: How does customers' previous repayment behaviour relate to their likelihood of defaulting in the following month?

In [ ]:
# Examine the repayment-status categories recorded in each period.
repayment_columns = [
    "PAY_0", "PAY_2", "PAY_3",
    "PAY_4", "PAY_5", "PAY_6"
]

for column in repayment_columns:
    print(f"\nCustomer counts for {column}:")
    print(df[column].value_counts().sort_index())

In [ ]:
# Calculate the number of customers and default rate for each repayment-status category.
summary_tables = {}

for column in repayment_columns:
    summary = df.groupby(column)[target_col].agg(
        Number_of_Customers="count",
        Default_Rate="mean"
    )
    summary["Default Rate (%)"] = summary["Default_Rate"] * 100
    summary = summary.drop(columns="Default_Rate")
    summary_tables[column] = summary

    print(f"\nDefault rate by {column}:")
    display(summary)

### Interpretation of the repayment-rate tables

Across the six repayment-status variables (`PAY_0`, `PAY_2`, `PAY_3`, `PAY_4`, `PAY_5` and `PAY_6`), categories representing more serious repayment delays generally show higher default rates in the following month. For example, status 2 recorded default rates of approximately 50.65% to 69.14% across the six periods, while status 3 recorded rates of approximately 57.50% to 75.78%. In comparison, status -1 generally recorded lower rates, approximately 15.59% to 16.99%.

These results suggest that previous repayment difficulties are associated with a greater likelihood of future default. Financial institutions may therefore use repayment history as one indicator when assessing credit risk. Categories with very small customer counts should be interpreted cautiously because their calculated rates may be less stable.

**Important:** The repayment-status codes are categories. In this dataset, -1 indicates payment duly, positive values indicate delays measured in months, and 0 and -2 are separate categories. They should not all be described as the same payment behaviour.

In [ ]:
# Compare default rates across repayment-status categories for all six periods.
# A grouped bar chart treats repayment statuses as categories rather than a continuous scale.
plot_data = []

for column in repayment_columns:
    rates = df.groupby(column)[target_col].mean() * 100
    for status, rate in rates.items():
        plot_data.append({
            "Repayment Status": status,
            "Repayment Period": column,
            "Default Rate (%)": rate
        })

plot_df = pd.DataFrame(plot_data)

plt.figure(figsize=(14, 7))
sns.barplot(
    data=plot_df,
    x="Repayment Status",
    y="Default Rate (%)",
    hue="Repayment Period"
)

plt.title("Default Rate by Previous Repayment Status")
plt.xlabel("Repayment Status Category")
plt.ylabel("Default Rate (%)")
plt.legend(title="Repayment Period", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

### Interpretation of the chart for Question 2

The grouped bar chart compares the default rate associated with each repayment-status category across the six repayment periods. It shows that categories associated with more serious delays generally have higher subsequent default rates, while status -1 generally has lower rates. This supports the conclusion that previous repayment behaviour is useful when assessing potential credit risk.

The chart shows an association, not proof that a particular repayment status causes default. Customer counts should also be considered when interpreting rates for less common categories.

## Question 4: How do credit limits differ between customers who default and those who do not?

In [ ]:
plt.figure(figsize=(8, 5))

sns.boxplot(
    x=target_col,
    y="LIMIT_BAL",
    data=df
)

plt.title("Credit Limit Distribution by Default Status")
plt.xlabel("Default Status (0 = No Default, 1 = Default)")
plt.ylabel("Credit Limit")
plt.tight_layout()
plt.show()

### Interpretation of Question 4

The box plot compares the distribution of credit limits between customers who defaulted and those who did not. Customers who did not default generally appear to have a higher median credit limit, and their middle 50% of credit limits also appears higher. However, both groups contain customers with high credit limits, showing that a high credit limit does not guarantee successful repayment.

Credit limit may be associated with default behaviour, but it should not be used as the only factor in assessing credit risk. Previous repayment behaviour, outstanding balances and credit utilisation should also be considered.

## Summary of Questions 1, 2 and 4

- Most customers in the dataset did not default (77.88%), but 22.12% did default in the following month.
- Previous repayment-status categories associated with more serious delays generally had higher subsequent default rates.
- Customers who did not default generally showed higher credit-limit medians in the box plot, although the two groups overlap.

Together, these findings indicate that default risk should be assessed using multiple indicators rather than credit limit alone. These are descriptive findings from this dataset and do not establish causation. The overall EDA conclusion should be written after the remaining EDA questions have also been reviewed.